# Shared BQ2 functions

Loading, scope and psychrometric functions used by all three BQ2 notebooks. Each notebook loads this file with `%run -i`.

The F5-Z1 flag records a **hypothesis** (sensor in the AC supply air); its placement and calibration have not been verified.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Where the shared project files live, relative to the project root (the folder holding data-cleaning/).
# If the team repo layout changes, update these lines only.
ZONE_HOURLY_CSV = "data-cleaning/outputs/aggregated/zone_hourly.csv"
WEATHER_CSV = "baseline/data/bangkok_era5_hourly_2018-2019.csv"
BASELINE_CODE_DIR = "analytics/baseline"

## Scope and flags

Floors 3–7, zones 1, 2, 4 and 5 (20 zones). Other floors/zones lack comparable sensors or layout. Retain F5-Z1 with an environmental QA flag, not an ordinary ranking. Environmental outages vary by channel; check actual coverage rather than assuming a whole season is usable.

In [ ]:
SCOPE_FLOORS = [3, 4, 5, 6, 7]
SCOPE_ZONES = [1, 2, 4, 5]

SUSPECT_SENSORS = {(5, 1): "Sensor in AC supply air: temperature biased cold, RH biased high (cleaning notebook 3.7/6.5)"}

## Psychrometrics

RH depends on temperature; humidity ratio expresses water vapour per kilogram of dry air. The functions provide dew point (°C), humidity ratio (g/kg), moist-air enthalpy (kJ/kg) and assumed-surface RH (%), using Magnus coefficients and 1013.25 hPa.

Assumed surface conditions are scenarios, not measurements of condensation or mould. Enthalpy is weather context, not measured AC cooling output.

In [ ]:
def sat_vp_hpa(t_c):
    """Saturation vapour pressure (hPa), Magnus formula (Sonntag 1990 coefficients)."""
    return 6.112 * np.exp(17.62 * t_c / (243.12 + t_c))


def dew_point_c(t_c, rh_pct):
    """Dew point (°C) from air temperature (°C) and relative humidity (%)."""
    rh = np.clip(rh_pct, 1, 100)
    g = np.log(rh / 100) + 17.62 * t_c / (243.12 + t_c)
    return 243.12 * g / (17.62 - g)


def humidity_ratio_gkg(t_c, rh_pct, p_hpa=1013.25):
    """Grams of water vapour per kg of dry air."""
    pv = np.clip(rh_pct, 0, 100) / 100 * sat_vp_hpa(t_c)
    return 622 * pv / (p_hpa - pv)


def enthalpy_kjkg(t_c, w_gkg):
    """Moist-air enthalpy (kJ per kg dry air). Captures sensible + latent load in one number."""
    w = w_gkg / 1000
    return 1.006 * t_c + w * (2501 + 1.86 * t_c)


def surface_rh_pct(dew_c, surface_t_c):
    """RH at a surface colder than the room air (same moisture content, lower temperature)."""
    return np.clip(100 * sat_vp_hpa(dew_c) / sat_vp_hpa(surface_t_c), 0, 100)

## Seasonal labels

Project month groups: hot March–May, rainy June–October, cool November–February. They are approximate categories; comparisons still require actual weather and adequate coverage.

In [ ]:
SEASON_ORDER = ["hot", "rainy", "cool"]


def season(month):
    """Hot: Mar-May, rainy: Jun-Oct, cool: Nov-Feb. Works on an int or a Series of months."""
    m = pd.Series(month) if not isinstance(month, pd.Series) else month
    out = pd.Series(np.select([m.between(3, 5), m.between(6, 10)], ["hot", "rainy"], "cool"), index=m.index)
    return pd.Categorical(out, categories=SEASON_ORDER, ordered=True)

## Loading and weather merge

`load_zone_hourly` filters scope and adds indoor features/flags. `merge_weather` adds outdoor features and the legacy `moisture_removal_gkg` field: outdoor minus indoor humidity ratio. That gap is a supporting indicator, not measured coil removal. Load the existing cache; do not refresh upstream data just to run BQ2.

In [ ]:
def load_zone_hourly(path, scope_only=True):
    """Cleaned zone x hour table + indoor dew point, humidity ratio, season and quality flag."""
    zh = pd.read_csv(path, parse_dates=["Date"])
    if scope_only:
        zh = zh[zh["floor"].isin(SCOPE_FLOORS) & zh["zone"].astype(int).isin(SCOPE_ZONES)].copy()
    zh["zone"] = zh["zone"].astype(int)
    zh["indoor_dew_C"] = dew_point_c(zh["temp_C"], zh["rh_pct"])
    zh["indoor_w_gkg"] = humidity_ratio_gkg(zh["temp_C"], zh["rh_pct"])
    zh["season"] = season(zh["Date"].dt.month)
    zh["year"] = zh["Date"].dt.year
    zh["suspect_sensor"] = [(f, z) in SUSPECT_SENSORS for f, z in zip(zh["floor"], zh["zone"])]
    return zh


def add_outdoor_psychrometrics(weather):
    """Adds outdoor humidity ratio and enthalpy to the ERA5 table from load_bangkok_era5_weather()."""
    w = weather.copy()
    w["outdoor_w_gkg"] = humidity_ratio_gkg(w["outdoor_temp_C"], w["outdoor_rh_pct"])
    w["outdoor_enthalpy_kJkg"] = enthalpy_kjkg(w["outdoor_temp_C"], w["outdoor_w_gkg"])
    return w


def merge_weather(zone_hourly, weather):
    """Joins hourly ERA5 onto the zone table by timestamp and adds moisture removal (out - in)."""
    w = add_outdoor_psychrometrics(weather)
    w = w.reset_index().rename(columns={w.index.name or "index": "Date"})
    df = zone_hourly.merge(w, on="Date", how="left")
    df["moisture_removal_gkg"] = df["outdoor_w_gkg"] - df["indoor_w_gkg"]
    return df

## AC on/off threshold

On = power above max(0.5 kW, 20% of the zone's 95th-percentile power). Used by the Track B screen (`bq2_ac.ipynb`).

In [ ]:
def ac_on_threshold(ac_kw):
    """'On' = above 20% of the zone's 95th-percentile AC power, and at least 0.5 kW (above standby)."""
    return max(0.5, 0.2 * ac_kw.quantile(0.95))